# 🦷 MIS RADIOGRAFÍAS — Construye tu propio dataset dental con IA
### Seminario: IA y Seguridad de Datos en Odontología
### Ponente: Marco Antonio Robles Huamán — Keluma SAC

---

## ¿Qué hace este notebook?

Toma **tus propias radiografías DICOM** de tu clínica dental y las convierte en un dataset listo para entrenar una IA.

## Pasos:

| # | Qué hacer |
|---|----------|
| 1️⃣ | Sube tus archivos `.dcm` a Google Drive en la carpeta que configures |
| 2️⃣ | Crea un Excel con el diagnóstico de cada radiografía (plantilla incluida) |
| 3️⃣ | Ejecuta **SETUP**, luego **CONFIGURAR**, luego **PROCESAR TODO** |
| 4️⃣ | Obtén tus PNGs nombrados + Excel catálogo + modelo IA entrenado con tus datos |

> ⚡ **Shift+Enter** para ejecutar cada celda

---

## 📁 Estructura de carpetas en tu Google Drive
```
Mi Drive/
└── mi_dataset_dental/
    ├── dicoms/          ← aquí van tus archivos .dcm
    │   ├── caso_001.dcm
    │   ├── caso_002.dcm
    │   └── ...
    ├── diagnosticos.xlsx  ← tu Excel con diagnósticos
    └── resultados/        ← aquí se guardan los PNGs y el catálogo (se crea solo)
```

---
# 1️⃣ SETUP — Ejecutar una sola vez

In [ ]:
print('📦 Instalando librerías...')
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'pydicom', 'pillow', 'openpyxl', 'tensorflow', '--quiet'], check=True)
print('✅ Librerías instaladas')

from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive conectado')
print()
print('━'*55)
print('  ✅ SETUP LISTO — configura el formulario de abajo')
print('━'*55)

---
# 2️⃣ CONFIGURAR — Indica dónde están tus archivos

In [ ]:
#@title ⚙️ CONFIGURACIÓN — Cambia solo estas opciones

#@markdown ### 📁 Rutas en tu Google Drive
CARPETA_DICOMS = "mi_dataset_dental/dicoms"  #@param {type:"string"}
#@markdown Carpeta donde están tus archivos `.dcm` (relativa a Mi Drive)

ARCHIVO_DIAGNOSTICOS = "mi_dataset_dental/diagnosticos.xlsx"  #@param {type:"string"}
#@markdown Excel con diagnósticos. Debe tener columnas: `archivo` y `diagnostico`
#@markdown Ejemplo: | caso_001.dcm | caries | · | caso_002.dcm | sano |

CARPETA_RESULTADOS = "mi_dataset_dental/resultados"  #@param {type:"string"}
#@markdown Carpeta donde se guardarán los PNGs y el catálogo (se crea automáticamente)

#@markdown ---
#@markdown ### ⚙️ Opciones de exportación
RESOLUCION_PX = 128  #@param [64, 128, 256]
#@markdown Resolución de las imágenes PNG exportadas

ANONIMIZAR = True  #@param {type:"boolean"}
#@markdown Eliminar datos del paciente antes de procesar (recomendado — HIPAA / Ley 29733)

ENTRENAR_MODELO = True  #@param {type:"boolean"}
#@markdown Entrenar un modelo de IA con tus propias radiografías (necesitas al menos 50 por categoría)

# Rutas completas
BASE_DRIVE = '/content/drive/MyDrive'
RUTA_DICOMS      = f'{BASE_DRIVE}/{CARPETA_DICOMS}'
RUTA_EXCEL_IN    = f'{BASE_DRIVE}/{ARCHIVO_DIAGNOSTICOS}'
RUTA_RESULTADOS  = f'{BASE_DRIVE}/{CARPETA_RESULTADOS}'

import os
os.makedirs(RUTA_RESULTADOS, exist_ok=True)
os.makedirs(f'{RUTA_RESULTADOS}/imagenes_png', exist_ok=True)
os.makedirs(f'{RUTA_RESULTADOS}/modelo', exist_ok=True)

print('✅ Configuración guardada:')
print(f'  📁 DICOMs       : {RUTA_DICOMS}')
print(f'  📗 Diagnósticos : {RUTA_EXCEL_IN}')
print(f'  📁 Resultados   : {RUTA_RESULTADOS}')
print(f'  🖼️  Resolución   : {RESOLUCION_PX}×{RESOLUCION_PX} px')
print(f'  🔒 Anonimizar   : {"Sí" if ANONIMIZAR else "No"}')
print(f'  🧠 Entrenar IA  : {"Sí" if ENTRENAR_MODELO else "No"}')
print()
print('▶ Ahora ejecuta la celda PROCESAR TODO')

---
# 3️⃣ PLANTILLA — Descarga el Excel de diagnósticos
> Si ya tienes tu Excel, sáltate esta celda.

In [ ]:
# ── GENERAR PLANTILLA EXCEL de diagnósticos ──────────────────
# Ejecuta esta celda para descargar la plantilla
# Llénala con tus archivos y diagnósticos y súbela a Drive

import pandas as pd, os

# Detectar archivos DICOM existentes si la carpeta ya existe
dicoms_encontrados = []
if os.path.exists(RUTA_DICOMS):
    dicoms_encontrados = [f for f in os.listdir(RUTA_DICOMS)
                          if f.lower().endswith('.dcm')]
    print(f'  ✅ Encontrados {len(dicoms_encontrados)} archivos DICOM en tu carpeta')

# Crear plantilla
if dicoms_encontrados:
    # Si ya hay DICOMs, usar sus nombres reales
    filas = [{'archivo': f, 'diagnostico': '', 'notas': ''}
             for f in sorted(dicoms_encontrados)]
else:
    # Plantilla de ejemplo
    filas = [
        {'archivo': 'caso_001.dcm', 'diagnostico': 'caries',      'notas': 'molar inferior derecho'},
        {'archivo': 'caso_002.dcm', 'diagnostico': 'sano',        'notas': ''},
        {'archivo': 'caso_003.dcm', 'diagnostico': 'periapical',  'notas': 'absceso leve'},
        {'archivo': 'caso_004.dcm', 'diagnostico': 'sano',        'notas': ''},
        {'archivo': 'caso_005.dcm', 'diagnostico': 'caries',      'notas': 'premolar'},
        {'archivo': 'caso_006.dcm', 'diagnostico': 'reabsorcion', 'notas': ''},
    ]

df_plantilla = pd.DataFrame(filas)

# Guardar en Drive
ruta_plantilla = f'{BASE_DRIVE}/mi_dataset_dental/diagnosticos.xlsx'
os.makedirs(os.path.dirname(ruta_plantilla), exist_ok=True)

with pd.ExcelWriter(ruta_plantilla, engine='openpyxl') as writer:
    df_plantilla.to_excel(writer, sheet_name='Diagnosticos', index=False)
    pd.DataFrame({
        'diagnostico': ['caries','sano','periapical','reabsorcion',
                        'fractura','impactado','periodontitis','otro'],
        'descripcion': [
            'Lesión cariosa — desmineralización del esmalte o dentina',
            'Sin patología visible',
            'Lesión periapical — infección en la punta de la raíz',
            'Reabsorción radicular interna o externa',
            'Fractura dental — corona o raíz',
            'Pieza dental retenida — no erupcionada',
            'Enfermedad periodontal — pérdida ósea alveolar',
            'Otro hallazgo — especificar en notas'
        ]
    }).to_excel(writer, sheet_name='Categorias', index=False)

print(f'✅ Plantilla guardada en Drive:')
print(f'   {ruta_plantilla}')
print()
if not dicoms_encontrados:
    print('📋 INSTRUCCIONES:')
    print('  1. Abre el archivo diagnosticos.xlsx en Drive')
    print('  2. Rellena la columna "diagnostico" con el diagnóstico de cada caso')
    print('  3. Sube tus archivos .dcm a la carpeta dicoms/')
    print('  4. Ejecuta la celda PROCESAR TODO')

---
# 4️⃣ PROCESAR TODO — Convierte, cataloga y entrena
> Presiona ▶ y espera. Todo se guarda en tu Drive.

In [ ]:
import pydicom, numpy as np, pandas as pd, cv2, os, time
import matplotlib.pyplot as plt
from PIL import Image
import warnings; warnings.filterwarnings('ignore')

AZUL='#0D1B40'; VERDE='#28a745'; ROJO='#dc3545'; CELESTE='#1A73E8'
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white'})

# ─── VERIFICAR ARCHIVOS ───────────────────────────────────────
print('▓'*55)
print('  VERIFICANDO ARCHIVOS')
print('▓'*55)

if not os.path.exists(RUTA_DICOMS):
    print(f'❌ No existe la carpeta: {RUTA_DICOMS}')
    print('   Crea la carpeta en Drive y sube tus archivos .dcm')
    raise FileNotFoundError(f'Carpeta no encontrada: {RUTA_DICOMS}')

archivos_dcm = sorted([f for f in os.listdir(RUTA_DICOMS) if f.lower().endswith('.dcm')])
print(f'  ✅ Archivos DICOM encontrados: {len(archivos_dcm)}')
for f in archivos_dcm[:5]: print(f'     · {f}')
if len(archivos_dcm)>5: print(f'     ... y {len(archivos_dcm)-5} más')

if not os.path.exists(RUTA_EXCEL_IN):
    print(f'⚠️  No existe el Excel de diagnósticos: {RUTA_EXCEL_IN}')
    print('   Ejecuta la celda PLANTILLA primero, llénala y súbela.')
    # Continuar sin diagnósticos — todos quedan como 'sin_etiquetar'
    df_dx = pd.DataFrame({'archivo': archivos_dcm,
                          'diagnostico': ['sin_etiquetar']*len(archivos_dcm),
                          'notas': ['']*len(archivos_dcm)})
else:
    df_dx = pd.read_excel(RUTA_EXCEL_IN)
    df_dx.columns = [c.lower().strip() for c in df_dx.columns]
    print(f'  ✅ Excel de diagnósticos: {len(df_dx)} filas')
    print(f'     Categorías: {df_dx["diagnostico"].unique().tolist()}')

# Índice de diagnósticos por nombre de archivo
dx_dict = dict(zip(df_dx['archivo'].astype(str), df_dx['diagnostico'].astype(str)))

# ─── PASO 1: ANONIMIZAR Y CONVERTIR DICOM → PNG ──────────────
print()
print('▓'*55)
print('  PASO 1: ANONIMIZAR + CONVERTIR DICOM → PNG')
print('▓'*55)

def anonimizar(dcm, id_caso):
    """Elimina datos personales — HIPAA / Ley 29733 Perú / LGPD Brasil"""
    dcm.PatientName      = 'ANONIMO'
    dcm.PatientID        = id_caso
    dcm.PatientBirthDate = ''
    dcm.PatientSex       = ''
    if hasattr(dcm, 'OtherPatientIDs'): dcm.OtherPatientIDs = ''
    return dcm

def dicom_a_array(dcm, resolucion):
    """DICOM → numpy array normalizado listo para PNG y Keras"""
    px = dcm.pixel_array
    if px.ndim == 3: px = px[:,:,0]           # tomar primer canal si hay varios
    pmin, pmax = px.min(), px.max()
    p = ((px-pmin)/(pmax-pmin)*255).astype(np.uint8) if pmax>pmin else px.astype(np.uint8)
    p = cv2.resize(p, (resolucion, resolucion))
    return p

registros = []
errores   = []
t0 = time.time()

for i, archivo in enumerate(archivos_dcm):
    ruta_dcm = f'{RUTA_DICOMS}/{archivo}'
    diagnostico = dx_dict.get(archivo, 'sin_etiquetar').lower().replace(' ','_')
    id_caso = f'CASO_{i+1:04d}'
    nombre_png = f'img_{i+1:04d}_{diagnostico}.png'
    ruta_png = f'{RUTA_RESULTADOS}/imagenes_png/{nombre_png}'

    try:
        dcm = pydicom.dcmread(ruta_dcm)
        nombre_paciente = str(getattr(dcm, 'PatientName', 'N/D'))

        if ANONIMIZAR:
            dcm = anonimizar(dcm, id_caso)

        arr = dicom_a_array(dcm, RESOLUCION_PX)
        Image.fromarray(arr, 'L').save(ruta_png)

        registros.append({
            'id':           i+1,
            'id_caso':      id_caso,
            'archivo_orig': archivo,
            'nombre_png':   nombre_png,
            'diagnostico':  diagnostico,
            'resolucion':   f'{RESOLUCION_PX}x{RESOLUCION_PX}',
            'anonimizado':  'Sí' if ANONIMIZAR else 'No',
            'notas':        df_dx.loc[df_dx['archivo']==archivo, 'notas'].values[0]
                            if 'notas' in df_dx.columns and archivo in df_dx['archivo'].values
                            else ''
        })
        print(f'  ✅ [{i+1}/{len(archivos_dcm)}] {archivo} → {nombre_png}')

    except Exception as e:
        errores.append({'archivo': archivo, 'error': str(e)})
        print(f'  ❌ [{i+1}/{len(archivos_dcm)}] {archivo}: {e}')

print(f'\n  ✅ {len(registros)} convertidos · ❌ {len(errores)} errores · {time.time()-t0:.0f}s')

df_cat = pd.DataFrame(registros)

# ─── PASO 2: GALERÍA VISUAL ───────────────────────────────────
if len(registros) > 0:
    print()
    print('▓'*55)
    print('  PASO 2: GALERÍA VISUAL')
    print('▓'*55)

    categorias = df_cat['diagnostico'].unique()
    N_COLS = min(5, len(registros))
    N_ROWS = len(categorias)

    fig, axes = plt.subplots(N_ROWS, N_COLS+1,
                             figsize=(3*(N_COLS+1), 3*N_ROWS))
    if N_ROWS == 1: axes = [axes]
    fig.suptitle('Mi Dataset Dental — Radiografías por Diagnóstico',
                 fontsize=13, fontweight='bold', color=AZUL)

    for fila, cat in enumerate(categorias):
        casos_cat = df_cat[df_cat['diagnostico']==cat]
        n_cat     = len(casos_cat)

        # Etiqueta
        axes[fila][0].set_facecolor('#eef2ff')
        axes[fila][0].text(0.5, 0.6, cat.replace('_',' ').title(),
                           ha='center', va='center', fontsize=11,
                           fontweight='bold', color=AZUL,
                           transform=axes[fila][0].transAxes)
        axes[fila][0].text(0.5, 0.3, f'{n_cat} caso{"s" if n_cat!=1 else ""}',
                           ha='center', va='center', fontsize=9, color='#555',
                           transform=axes[fila][0].transAxes)
        axes[fila][0].axis('off')

        # Imágenes
        muestra = casos_cat.head(N_COLS)
        for col, (_, row) in enumerate(muestra.iterrows()):
            ruta = f'{RUTA_RESULTADOS}/imagenes_png/{row["nombre_png"]}'
            if os.path.exists(ruta):
                img = np.array(Image.open(ruta))
                axes[fila][col+1].imshow(img, cmap='gray')
                axes[fila][col+1].set_title(row['id_caso'], fontsize=7)
            axes[fila][col+1].axis('off')
        for col in range(len(muestra), N_COLS):
            axes[fila][col+1].axis('off')

    plt.tight_layout()
    ruta_gal = f'{RUTA_RESULTADOS}/galeria_mi_dataset.png'
    fig.savefig(ruta_gal, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'  💾 galeria_mi_dataset.png guardada')

# ─── PASO 3: EXCEL CATÁLOGO ───────────────────────────────────
print()
print('▓'*55)
print('  PASO 3: CATÁLOGO EXCEL')
print('▓'*55)

ruta_excel_out = f'{RUTA_RESULTADOS}/catalogo_mi_dataset.xlsx'
with pd.ExcelWriter(ruta_excel_out, engine='openpyxl') as writer:
    df_cat.to_excel(writer, sheet_name='Catalogo', index=False)
    print('  ✅ Hoja 1: Catálogo completo')

    resumen = df_cat.groupby('diagnostico').size().reset_index(name='cantidad')
    resumen['porcentaje'] = (resumen['cantidad']/len(df_cat)*100).round(1)
    resumen.to_excel(writer, sheet_name='Resumen por diagnostico', index=False)
    print('  ✅ Hoja 2: Resumen por diagnóstico')

    if errores:
        pd.DataFrame(errores).to_excel(writer, sheet_name='Errores', index=False)
        print(f'  ⚠️  Hoja 3: {len(errores)} errores')

    pd.DataFrame({
        'Término': ['DICOM','PNG','Anonimización','HIPAA','Ley 29733','LGPD',
                    'Dataset','Transfer Learning','Grad-CAM'],
        'Definición': [
            'Formato estándar de imágenes médicas — contiene imagen + datos del paciente',
            'Formato de imagen sin pérdida — recomendado para IA médica',
            'Eliminación de datos personales del paciente antes de procesar',
            'Health Insurance Portability and Accountability Act — EE.UU.',
            'Ley de Protección de Datos Personales — Perú',
            'Lei Geral de Proteção de Dados — Brasil',
            'Conjunto de imágenes etiquetadas para entrenar una IA',
            'Reutilizar un modelo preentrenado y adaptarlo con tus datos',
            'Mapa visual que muestra dónde mira la IA para diagnosticar'
        ]
    }).to_excel(writer, sheet_name='Glosario', index=False)
    print('  ✅ Hoja 3: Glosario')

print(f'  💾 catalogo_mi_dataset.xlsx guardado')

# ─── PASO 4: ENTRENAR MODELO (opcional) ───────────────────────
if ENTRENAR_MODELO and len(df_cat) >= 10:
    print()
    print('▓'*55)
    print('  PASO 4: ENTRENAR MODELO DE IA CON TUS DATOS')
    print('▓'*55)

    from tensorflow import keras
    from tensorflow.keras import layers
    import tensorflow as tf

    categorias_unicas = sorted(df_cat['diagnostico'].unique())
    cat_a_idx = {c:i for i,c in enumerate(categorias_unicas)}
    n_clases  = len(categorias_unicas)

    print(f'  Clases detectadas ({n_clases}): {categorias_unicas}')

    # Cargar imágenes PNG
    X, y_labels = [], []
    for _, row in df_cat.iterrows():
        ruta_png = f'{RUTA_RESULTADOS}/imagenes_png/{row["nombre_png"]}'
        if os.path.exists(ruta_png):
            img = np.array(Image.open(ruta_png).resize((RESOLUCION_PX,RESOLUCION_PX)))
            rgb = np.stack([img,img,img], axis=-1).astype(np.float32)/255.0
            X.append(rgb)
            y_labels.append(cat_a_idx[row['diagnostico']])

    X = np.array(X); y = np.array(y_labels)
    print(f'  Dataset: {len(X)} imágenes · {n_clases} clases')

    if len(X) < 10:
        print('  ⚠️  Necesitas al menos 10 imágenes para entrenar. Agrega más DICOMs.')
    else:
        # Split train/test
        idx_all = np.random.permutation(len(X))
        n_train = int(len(X)*0.8)
        X_tr, y_tr = X[idx_all[:n_train]], y[idx_all[:n_train]]
        X_te, y_te = X[idx_all[n_train:]], y[idx_all[n_train:]]

        res = RESOLUCION_PX
        if n_clases == 2:
            # Clasificación binaria
            base = keras.applications.MobileNetV2(
                input_shape=(res,res,3), include_top=False, weights='imagenet')
            base.trainable = False
            model = keras.Sequential([
                base, layers.GlobalAveragePooling2D(),
                layers.Dense(64, activation='relu'), layers.Dropout(0.3),
                layers.Dense(1, activation='sigmoid')
            ])
            model.compile(optimizer='adam', loss='binary_crossentropy',
                          metrics=['accuracy'])
            model.build((None,res,res,3))
            y_tr_m, y_te_m = y_tr, y_te
        else:
            # Clasificación multiclase
            base = keras.applications.MobileNetV2(
                input_shape=(res,res,3), include_top=False, weights='imagenet')
            base.trainable = False
            model = keras.Sequential([
                base, layers.GlobalAveragePooling2D(),
                layers.Dense(128, activation='relu'), layers.Dropout(0.3),
                layers.Dense(n_clases, activation='softmax')
            ])
            model.compile(optimizer='adam',
                          loss='sparse_categorical_crossentropy',
                          metrics=['accuracy'])
            model.build((None,res,res,3))
            y_tr_m, y_te_m = y_tr, y_te

        print(f'  Entrenando...')
        h = model.fit(X_tr, y_tr_m, epochs=10, batch_size=8,
                      validation_data=(X_te, y_te_m), verbose=1,
                      callbacks=[keras.callbacks.EarlyStopping(
                          patience=3, restore_best_weights=True)])

        ruta_model = f'{RUTA_RESULTADOS}/modelo/modelo_mis_radiografias.keras'
        model.save(ruta_model)
        _, acc = model.evaluate(X_te, y_te_m, verbose=0)
        print(f'  ✅ Accuracy en prueba: {acc*100:.1f}%')
        print(f'  💾 Modelo guardado: {ruta_model}')

        # Curvas
        fig, ax = plt.subplots(1,1,figsize=(8,4))
        ax.plot(h.history['accuracy'],'o-',color=CELESTE,label='Entrenamiento')
        ax.plot(h.history['val_accuracy'],'s--',color=ROJO,label='Validación')
        ax.set_title(f'Mi Modelo — Accuracy final: {acc*100:.1f}%',fontsize=12,
                     fontweight='bold',color=AZUL)
        ax.legend(); ax.grid(True,alpha=0.3); ax.set_ylim([0,1.05])
        plt.tight_layout()
        fig.savefig(f'{RUTA_RESULTADOS}/curvas_mi_modelo.png',
                    dpi=150,bbox_inches='tight',facecolor='white')
        plt.show()

elif ENTRENAR_MODELO:
    print('  ⚠️  Necesitas más imágenes para entrenar el modelo.')

# ─── RESUMEN FINAL ────────────────────────────────────────────
print()
print('▓'*55)
print('  ✅ PROCESO COMPLETADO')
print('▓'*55)
print(f'  📁 Resultados en: {RUTA_RESULTADOS}/')
print()
for f in sorted(os.listdir(RUTA_RESULTADOS)):
    ruta_f = f'{RUTA_RESULTADOS}/{f}'
    if os.path.isfile(ruta_f):
        kb = os.path.getsize(ruta_f)/1024
        print(f'  📄 {f:<40} {kb:>6.0f} KB')
    elif os.path.isdir(ruta_f):
        n = len(os.listdir(ruta_f))
        print(f'  📁 {f}/ ({n} archivos)')
print()
print('  Diagnósticos en tu dataset:')
for _, row in resumen.iterrows():
    print(f'  · {row["diagnostico"]:<20}: {row["cantidad"]:>3} casos ({row["porcentaje"]}%)')
print()
print('━'*55)
print('  🎓 Seminario IA y Seguridad de Datos en Odontología')
print('  Marco Antonio Robles Huamán — Keluma SAC')
print('━'*55)